In [ ]:
# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================
import pandas as pd                      # Data manipulation and analysis
import numpy as np                       # Numerical operations
import matplotlib.pyplot as plt          # Plotting graphs
import seaborn as sns                    # Data visualization
from sklearn.model_selection import train_test_split   # Split data into train/test
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # Evaluation metrics
from sklearn.linear_model import LinearRegression      # Linear Regression model
from sklearn.ensemble import GradientBoostingRegressor # Gradient Boosting model

# Mount Google Drive to access dataset
from google.colab import drive
drive.mount('/content/drive')

# ==========================================
# 2. LOAD & CLEAN DATA
# ==========================================
df = pd.read_csv("/content/drive/MyDrive/House Price Prediction Dataset.csv")  # Load dataset
df = df.drop('Id', axis=1)  # Remove ID column (not useful for prediction)

# ==========================================
# 🔍 EDA (PRINT-BASED DATA ANALYSIS ONLY)
# ==========================================

# Show first 5 rows
print("\n--- FIRST 5 ROWS ---")
print(df.head())

# Show last 5 rows
print("\n--- LAST 5 ROWS ---")
print(df.tail())

# Show dataset shape (rows, columns)
print("\n--- DATA SHAPE ---")
print("Rows:", df.shape[0], "Columns:", df.shape[1])

# Show column names
print("\n--- COLUMN NAMES ---")
print(df.columns)

# Show data types and non-null values
print("\n--- DATA INFO ---")
df.info()

# Statistical summary (mean, std, min, max)
print("\n--- STATISTICAL SUMMARY ---")
print(df.describe(include='all'))

# Check missing values in each column
print("\n--- MISSING VALUES ---")
print(df.isnull().sum())

# Check duplicate rows
print("\n--- DUPLICATES ---")
print("Duplicate rows:", df.duplicated().sum())

# ==========================================
# 3. PREPROCESSING & FEATURE ENGINEERING
# ==========================================
# Instructions: Preprocess square footage, bedrooms, and location.

# Feature Engineering
df['house_age'] = 2026 - df['YearBuilt']  # Create new feature: age of house
df['total_rooms'] = df['Bedrooms'] + df['Bathrooms']  # Total rooms feature

# Encoding Categorical Data
df['Garage'] = df['Garage'].map({'Yes': 1, 'No': 0})  # Convert Yes/No to 1/0

condition_map = {'Poor':1, 'Fair':2, 'Good':3, 'Excellent':4}  # Map condition categories
df['Condition'] = df['Condition'].map(condition_map)  # Apply mapping

# One-Hot Encoding for Location (convert categorical to multiple binary columns)
df = pd.get_dummies(df, columns=['Location'], drop_first=True)

df = df.drop('YearBuilt', axis=1)  # Remove original YearBuilt column after feature creation

# ==========================================
# 4. FEATURES & TARGET SPLIT
# ==========================================
X = df.drop('Price', axis=1)  # Features (independent variables)
y = df['Price']               # Target variable (dependent variable)
# ==========================================
# 🎯 FEATURES & TARGET ANALYSIS
# ==========================================

# Define target variable
target = 'Price'

# Define features
features = df.drop(target, axis=1).columns

print("\n--- TARGET VARIABLE ---")
print(target)

print("\n--- FEATURES (INDEPENDENT VARIABLES) ---")
print(list(features))

# Show feature data
print("\n--- FEATURES DATA (FIRST 5 ROWS) ---")
print(df[features].head())

# Show target data
print("\n--- TARGET DATA (FIRST 5 ROWS) ---")
print(df[target].head())

# Correlation with target
print("\n--- CORRELATION WITH TARGET (Price) ---")
print(df.corr(numeric_only=True)['Price'].sort_values(ascending=False))

# Split dataset into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# 5. MODEL TRAINING (Linear & Gradient Boosting)
# ==========================================

# Linear Regression model training
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Gradient Boosting model training (main model)
gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)

# ==========================================
# 6. PREDICTIONS & EVALUATION (MAE, RMSE, R2)
# ==========================================

# Predict values using Gradient Boosting model
y_pred = gb_model.predict(X_test)

print("--- MODEL EVALUATION ---")
print("MAE  (Mean Absolute Error):", mean_absolute_error(y_test, y_pred))  # Average absolute error
print("RMSE (Root Mean Squared Error):", np.sqrt(mean_squared_error(y_test, y_pred)))  # Penalizes large errors
print("R2 Score:", r2_score(y_test, y_pred))  # Model performance score

# ==========================================
# 7. VISUALIZATION (Actual vs Predicted)
# ==========================================

import seaborn as sns
import matplotlib.pyplot as plt

# Correlation heatmap
plt.figure(figsize=(10,10))
sns.heatmap(df.corr(numeric_only=True), annot=True)  # Show correlation between features
plt.show()
plt.tight_layout()

# Actual vs Predicted scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5, color='blue')  # Plot predictions vs actual values

# Reference line (perfect prediction line)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Prices (Gradient Boosting)")
plt.show()